In [1]:
import json
from pathlib import Path

archivo_enero = Path("../data/raw/2025/releases_2025_enero_subasta_inversa_electronica.json")

with open(archivo_enero, "r", encoding="utf-8") as archivo:
    datos_enero = json.load(archivo)

type(datos_enero)

list

In [2]:
len(datos_enero)

91

#### Identificar el tipo de cada registro

Se analiza el primer elemento de la lista para verificar si cada registro está almacenado como un diccionario de Python.

In [7]:
type(datos_enero[0])

dict

#### Identificar los campos disponibles

Se muestran las claves del primer registro para conocer qué variables contiene la estructura del archivo JSON.

In [6]:
datos_enero[0].keys()

dict_keys(['uri', 'license', 'version', 'releases', 'publisher', 'extensions', 'publishedDate', 'publicationPolicy'])

#### Explorar la lista de releases

Se accede a la clave `releases`, donde se encuentran los registros individuales de contratación pública contenidos en el paquete.

In [5]:
type(datos_enero[0]["releases"])

list

In [8]:
len(datos_enero[0]["releases"])

1

#### Identificar los campos del primer release

Se accede al primer registro de contratación para conocer las variables disponibles dentro de cada procedimiento publicado.

In [9]:
datos_enero[0]["releases"][0].keys()

dict_keys(['id', 'tag', 'date', 'ocid', 'buyer', 'tender', 'parties', 'language', 'planning', 'initiationType'])

#### Explorar la entidad compradora

Se revisa la estructura del campo `buyer` para identificar la entidad contratante y los datos disponibles para construir los nodos de compradores.

In [10]:
datos_enero[0]["releases"][0]["buyer"]

{'id': 'EC-RUC-1560001400001-77741', 'name': 'Municipio El Chaco'}

#### Explorar la información del procedimiento

Se revisa la estructura del campo `tender` para identificar datos como el código del proceso, estado, descripción, monto y método de contratación.

In [11]:
datos_enero[0]["releases"][0]["tender"].keys()

dict_keys(['id', 'lots', 'title', 'status', 'enquiries', 'tenderers', 'awardPeriod', 'description', 'hasEnquiries', 'tenderPeriod', 'awardCriteria', 'enquiryPeriod', 'procuringEntity', 'numberOfTenderers', 'procurementMethod', 'mainProcurementCategory', 'procurementMethodDetails'])

#### Revisar los valores principales del procedimiento

Se consultan los campos más relevantes del primer procedimiento para conocer cómo están representados el identificador, título, estado, método de contratación y número de oferentes.

In [12]:
tender = datos_enero[0]["releases"][0]["tender"]

{
    "id": tender.get("id"),
    "title": tender.get("title"),
    "status": tender.get("status"),
    "procurementMethod": tender.get("procurementMethod"),
    "procurementMethodDetails": tender.get("procurementMethodDetails"),
    "numberOfTenderers": tender.get("numberOfTenderers")
}

{'id': 'SIE-CHACO-2025-009-77741',
 'title': 'SIE-CHACO-2025-009-77741',
 'status': 'active',
 'procurementMethod': 'open',
 'procurementMethodDetails': 'Subasta Inversa Electrónica',
 'numberOfTenderers': 4}

#### Explorar los oferentes del procedimiento

Se revisa el campo `tenderers` para identificar a los proveedores que participaron en el procedimiento y los datos disponibles para construir los nodos de oferentes.

In [13]:
type(tender["tenderers"]), len(tender["tenderers"])

(list, 4)

#### Identificar los campos de los oferentes

Se analiza el primer oferente para conocer qué datos están disponibles sobre los proveedores participantes.

In [14]:
tender["tenderers"][0].keys()

dict_keys(['id', 'name'])

#### Visualizar los oferentes participantes

Se muestran los identificadores y nombres de todos los oferentes del primer procedimiento para verificar cómo se representan los proveedores participantes.

In [15]:
tender["tenderers"]

[{'id': 'EC-RUC-1600412769001-311697', 'name': 'ALVEAR AREVALO DIEGO ANDRES'},
 {'id': 'EC-RUC-1600537151001-440152', 'name': 'SOLIS ROBALINO MELANI MAYTE'},
 {'id': 'EC-RUC-1792131480001-80578', 'name': 'RAMFORTRADE CIA. LTDA.'},
 {'id': 'EC-RUC-1792734789001-822600',
  'name': 'COMERCIALIZADORA TRACTOR-ZONE CIA LTDA'}]

#### Explorar las partes relacionadas

Se revisa el campo `parties` para identificar las organizaciones involucradas en el procedimiento y los roles que desempeñan.

In [16]:
release = datos_enero[0]["releases"][0]

type(release["parties"]), len(release["parties"])

(list, 5)

#### Identificar la estructura de las partes

Se analiza la primera organización de la lista `parties` para conocer sus identificadores, nombre y roles dentro del procedimiento.

In [17]:
release["parties"][0].keys()

dict_keys(['id', 'name', 'roles', 'address', 'identifier', 'contactPoint'])

#### Identificar los roles de las organizaciones

Se muestran el identificador, nombre y roles de cada organización para distinguir a la entidad compradora de los proveedores participantes.

In [1]:
[
    {
        "id": parte.get("id"),
        "name": parte.get("name"),
        "roles": parte.get("roles")
    }
    for parte in release["parties"]
]

NameError: name 'release' is not defined

#### Verificar la existencia de información de adjudicación

Se revisan las claves de todos los releases para determinar si existe el campo `awards`, donde normalmente se identifica al proveedor adjudicado.

In [19]:
claves_releases = set()

for paquete in datos_enero:
    for release_item in paquete.get("releases", []):
        claves_releases.update(release_item.keys())

sorted(claves_releases)

['auctions',
 'awards',
 'buyer',
 'contracts',
 'date',
 'id',
 'initiationType',
 'language',
 'ocid',
 'parties',
 'planning',
 'tag',
 'tender']

#### Contar los releases con información de adjudicación

Se cuenta cuántos releases de enero contienen el campo `awards`, con el fin de identificar los procedimientos que ya incluyen información sobre proveedores adjudicados.

In [20]:
releases_con_awards = []

for paquete in datos_enero:
    for release_item in paquete.get("releases", []):
        if release_item.get("awards"):
            releases_con_awards.append(release_item)

len(releases_con_awards)

78

#### Explorar la estructura de las adjudicaciones

Se analiza el primer release que contiene información de adjudicación para identificar los campos disponibles sobre el proveedor ganador, el monto y el estado de la adjudicación.

In [21]:
releases_con_awards[0]["awards"]

[{'id': '2428016-SIE-HEP-2024-00115',
  'date': '2025-02-28T20:00:35-05:00',
  'items': [{'id': '4263344-DS-SO',
    'unit': {'id': '436', 'name': 'Unidad', 'scheme': 'SERCOP'},
    'quantity': 6600,
    'description': 'INSUMOS DE USO GENERAL',
    'classification': {'id': '352901091',
     'scheme': 'CPC',
     'description': 'INSUMOS DE USO GENERAL'},
    'additionalClassifications': [{'id': '35290.10.9',
      'uri': 'https://www.compraspublicas.gob.ec/ProcesoContratacion/compras/exe/verComProductos_exe.php?tipo=buscar&idProducto=35290.10.9',
      'scheme': 'CPC',
      'description': 'INSUMOS MEDICOS HOSPITALARIOS'}]}],
  'value': {'amount': 243500, 'currency': 'USD'},
  'suppliers': [{'id': 'EC-RUC-1312665100001-1217500',
    'name': 'ZAMBRANO VELEZ JEAN PIERRE'}],
  'description': 'Se resuelve ADJUDICAR el proceso de Subasta Inversa Electrónica Nro. SIE-HEP-2024-00115, cuyo objeto es la "ADQUISICIÓN DE DISPOSITIVOS MÉDICOS - EQUIPO DE PROTECCIÓN PERSONAL PARA LA ATENCIÓN OPORTUN

#### Extraer los datos principales de la adjudicación

Se seleccionan el identificador de la adjudicación, la fecha, el monto, la moneda y el proveedor adjudicado para comprobar cómo se estructurarán las relaciones de adjudicación en el modelo de red.

In [22]:
award = releases_con_awards[0]["awards"][0]

{
    "award_id": award.get("id"),
    "fecha": award.get("date"),
    "monto": award.get("value", {}).get("amount"),
    "moneda": award.get("value", {}).get("currency"),
    "proveedores": award.get("suppliers", [])
}

{'award_id': '2428016-SIE-HEP-2024-00115',
 'fecha': '2025-02-28T20:00:35-05:00',
 'monto': 243500,
 'moneda': 'USD',
 'proveedores': [{'id': 'EC-RUC-1312665100001-1217500',
   'name': 'ZAMBRANO VELEZ JEAN PIERRE'}]}

#### Contar procedimientos únicos mediante el OCID

Se recopilan los identificadores `ocid` de todos los releases para determinar cuántos procedimientos de contratación únicos existen y verificar si un mismo procedimiento aparece en varias etapas.

In [23]:
todos_los_releases = [
    release_item
    for paquete in datos_enero
    for release_item in paquete.get("releases", [])
]

ocids = [
    release_item.get("ocid")
    for release_item in todos_los_releases
    if release_item.get("ocid")
]

{
    "total_releases": len(todos_los_releases),
    "procedimientos_unicos": len(set(ocids)),
    "releases_repetidos": len(ocids) - len(set(ocids))
}

{'total_releases': 91, 'procedimientos_unicos': 91, 'releases_repetidos': 0}

#### Contar proveedores adjudicados por procedimiento

Se analiza cuántos proveedores aparecen en cada adjudicación para identificar si los procedimientos tienen un único ganador o varios proveedores adjudicados.

In [24]:
cantidad_proveedores_por_award = []

for release_item in releases_con_awards:
    for award_item in release_item.get("awards", []):
        cantidad_proveedores_por_award.append(
            len(award_item.get("suppliers", []))
        )

{
    "total_awards": len(cantidad_proveedores_por_award),
    "minimo_proveedores": min(cantidad_proveedores_por_award),
    "maximo_proveedores": max(cantidad_proveedores_por_award),
    "distribucion": {
        cantidad: cantidad_proveedores_por_award.count(cantidad)
        for cantidad in sorted(set(cantidad_proveedores_por_award))
    }
}

{'total_awards': 78,
 'minimo_proveedores': 1,
 'maximo_proveedores': 1,
 'distribucion': {1: 78}}

#### Analizar los montos adjudicados

Se extraen los montos de todas las adjudicaciones para conocer el valor mínimo, máximo, promedio y total adjudicado durante el período analizado.

In [25]:
montos_adjudicados = []

for release_item in releases_con_awards:
    for award_item in release_item.get("awards", []):
        monto = award_item.get("value", {}).get("amount")

        if monto is not None:
            montos_adjudicados.append(monto)

{
    "cantidad_montos": len(montos_adjudicados),
    "monto_minimo": min(montos_adjudicados),
    "monto_maximo": max(montos_adjudicados),
    "monto_promedio": sum(montos_adjudicados) / len(montos_adjudicados),
    "monto_total": sum(montos_adjudicados)
}

{'cantidad_montos': 78,
 'monto_minimo': 7430.71,
 'monto_maximo': 474000,
 'monto_promedio': 80329.25833333333,
 'monto_total': 6265682.15}

In [4]:
from pathlib import Path
import json
import pandas as pd

# 1. Localizar la carpeta con los 12 archivos
rutas_posibles = [
    Path.cwd() / "data" / "raw" / "2025",
    Path.cwd().parent / "data" / "raw" / "2025"
]

carpeta_datos = next(
    (ruta for ruta in rutas_posibles if ruta.exists()),
    None
)

if carpeta_datos is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/raw/2025."
    )

archivos = sorted(carpeta_datos.glob("*.json"))

print("Carpeta encontrada:", carpeta_datos)
print("Número de archivos JSON:", len(archivos))

if len(archivos) != 12:
    print("ADVERTENCIA: Se esperaban 12 archivos mensuales.")


# 2. Conjuntos para evitar contar identificadores repetidos
ocids = set()
entidades = set()
oferentes = set()
proveedores_adjudicados = set()
procedimientos_con_adjudicacion = set()
adjudicaciones = set()

total_releases = 0
resumen_mensual = []


def identificador_actor(actor):
    """Obtiene el identificador; usa el nombre solo si no existe ID."""
    if not isinstance(actor, dict):
        return None

    return actor.get("id") or actor.get("name")


# 3. Recorrer los 12 archivos
for archivo in archivos:
    with open(archivo, "r", encoding="utf-8") as entrada:
        contenido = json.load(entrada)

    paquetes = contenido if isinstance(contenido, list) else [contenido]

    releases_archivo = 0
    ocids_archivo = set()

    for paquete in paquetes:
        if not isinstance(paquete, dict):
            continue

        releases = paquete.get("releases", [])

        # Por si algún archivo contiene directamente un release
        if not releases and paquete.get("ocid"):
            releases = [paquete]

        for release in releases:
            total_releases += 1
            releases_archivo += 1

            ocid = release.get("ocid")
            if ocid:
                ocids.add(ocid)
                ocids_archivo.add(ocid)

            tender = release.get("tender") or {}

            # Entidad compradora o contratante
            for actor in [
                release.get("buyer"),
                tender.get("procuringEntity")
            ]:
                actor_id = identificador_actor(actor)
                if actor_id:
                    entidades.add(actor_id)

            # Oferentes registrados en tender
            for oferente in tender.get("tenderers", []) or []:
                oferente_id = identificador_actor(oferente)
                if oferente_id:
                    oferentes.add(oferente_id)

            # Roles registrados en parties
            for participante in release.get("parties", []) or []:
                participante_id = identificador_actor(participante)
                roles = participante.get("roles", []) or []

                if participante_id and (
                    "buyer" in roles or "procuringEntity" in roles
                ):
                    entidades.add(participante_id)

                if participante_id and "tenderer" in roles:
                    oferentes.add(participante_id)

            # Adjudicaciones y proveedores adjudicados
            for adjudicacion in release.get("awards", []) or []:
                if ocid:
                    procedimientos_con_adjudicacion.add(ocid)

                award_id = adjudicacion.get("id")

                if award_id:
                    adjudicaciones.add((ocid, award_id))

                for proveedor in adjudicacion.get("suppliers", []) or []:
                    proveedor_id = identificador_actor(proveedor)

                    if proveedor_id:
                        proveedores_adjudicados.add(proveedor_id)

    resumen_mensual.append({
        "archivo": archivo.name,
        "releases": releases_archivo,
        "procedimientos_unicos_ocid": len(ocids_archivo)
    })


# 4. Presentar el resumen mensual
tabla_mensual = pd.DataFrame(resumen_mensual)
display(tabla_mensual)


# 5. Presentar el inventario general
resumen_general = {
    "archivos_json": len(archivos),
    "releases_totales": total_releases,
    "procedimientos_unicos_ocid": len(ocids),
    "entidades_identificadas": len(entidades),
    "oferentes_unicos": len(oferentes),
    "procedimientos_con_adjudicacion": len(
        procedimientos_con_adjudicacion
    ),
    "adjudicaciones_unicas": len(adjudicaciones),
    "proveedores_adjudicados_unicos": len(
        proveedores_adjudicados
    )
}

display(
    pd.DataFrame.from_dict(
        resumen_general,
        orient="index",
        columns=["cantidad"]
    )
)


# 6. Generar dos párrafos preliminares
print(
    f"\nEl conjunto inicial de datos estuvo integrado por "
    f"{len(archivos)} archivos mensuales en formato JSON. "
    f"En total se identificaron {total_releases:,} releases, "
    f"correspondientes a {len(ocids):,} procedimientos únicos "
    f"según el identificador ocid."
)

print(
    f"\nEn la revisión preliminar se reconocieron "
    f"{len(entidades):,} entidades contratantes y "
    f"{len(oferentes):,} oferentes. Asimismo, "
    f"{len(procedimientos_con_adjudicacion):,} procedimientos "
    f"presentaron información de adjudicación, con "
    f"{len(proveedores_adjudicados):,} proveedores adjudicados "
    f"identificados."
)

Carpeta encontrada: /Users/dianaaltamirano/PycharmProjects/analitica_redes_contratacion_ecuador_2025/data/raw/2025
Número de archivos JSON: 12


,archivo,releases,procedimientos_unicos_ocid
0,releases_2025_abril_subasta_inversa_electronic...,2473,2473
1,releases_2025_agosto_subasta_inversa_electroni...,787,787
2,releases_2025_diciembre_subasta_inversa_electr...,2214,2214
3,releases_2025_enero_subasta_inversa_electronic...,91,91
4,releases_2025_febrero_subasta_inversa_electron...,732,732
5,releases_2025_julio_subasta_inversa_electronic...,2385,2385
6,releases_2025_junio_subasta_inversa_electronic...,1946,1946
7,releases_2025_marzo_subasta_inversa_electronic...,1661,1661
8,releases_2025_mayo_subasta_inversa_electronica...,1855,1855
9,releases_2025_noviembre_subasta_inversa_electr...,1566,1566


,cantidad
archivos_json,12
releases_totales,18326
procedimientos_unicos_ocid,18326
entidades_identificadas,1737
oferentes_unicos,12125
procedimientos_con_adjudicacion,15718
adjudicaciones_unicas,15718
proveedores_adjudicados_unicos,5969



El conjunto inicial de datos estuvo integrado por 12 archivos mensuales en formato JSON. En total se identificaron 18,326 releases, correspondientes a 18,326 procedimientos únicos según el identificador ocid.

En la revisión preliminar se reconocieron 1,737 entidades contratantes y 12,125 oferentes. Asimismo, 15,718 procedimientos presentaron información de adjudicación, con 5,969 proveedores adjudicados identificados.


In [2]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import json

ruta = "../data/raw/2025/releases_2025_enero_subasta_inversa_electronica.json"

with open(ruta, "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

type(datos)


list

In [2]:
len(datos)


91

In [3]:
datos[0].keys()

dict_keys(['uri', 'license', 'version', 'releases', 'publisher', 'extensions', 'publishedDate', 'publicationPolicy'])

In [4]:
len(datos[0]["releases"])

1

In [5]:
datos[0]["releases"][0].keys()

dict_keys(['id', 'tag', 'date', 'ocid', 'buyer', 'tender', 'parties', 'language', 'planning', 'initiationType'])

In [6]:
datos[0]["releases"][0]["ocid"]

'ocds-5wno2w-SIE-CHACO-2025-009-77741'

In [7]:
datos[0]["releases"][0]["buyer"]


{'id': 'EC-RUC-1560001400001-77741', 'name': 'Municipio El Chaco'}

In [9]:
datos[0]["releases"][0]["tender"].keys()

dict_keys(['id', 'lots', 'title', 'status', 'enquiries', 'tenderers', 'awardPeriod', 'description', 'hasEnquiries', 'tenderPeriod', 'awardCriteria', 'enquiryPeriod', 'procuringEntity', 'numberOfTenderers', 'procurementMethod', 'mainProcurementCategory', 'procurementMethodDetails'])

In [10]:
datos[0]["releases"][0]["tender"]["numberOfTenderers"]

4

In [11]:
len(datos[0]["releases"][0]["tender"]["tenderers"])

4

In [12]:
datos[0]["releases"][0]["tender"]["tenderers"]

[{'id': 'EC-RUC-1600412769001-311697', 'name': 'ALVEAR AREVALO DIEGO ANDRES'},
 {'id': 'EC-RUC-1600537151001-440152', 'name': 'SOLIS ROBALINO MELANI MAYTE'},
 {'id': 'EC-RUC-1792131480001-80578', 'name': 'RAMFORTRADE CIA. LTDA.'},
 {'id': 'EC-RUC-1792734789001-822600',
  'name': 'COMERCIALIZADORA TRACTOR-ZONE CIA LTDA'}]

In [13]:
datos[0]["releases"][0].keys()

dict_keys(['id', 'tag', 'date', 'ocid', 'buyer', 'tender', 'parties', 'language', 'planning', 'initiationType'])

In [14]:
"awards" in datos[0]["releases"][0]

False

In [15]:
for i, paquete in enumerate(datos):
    release = paquete["releases"][0]
    if "awards" in release:
        print("Índice:", i)
        print("OCID:", release["ocid"])
        break

Índice: 2
OCID: ocds-5wno2w-SIE-HEP-2024-00115-889479


In [16]:
datos[2]["releases"][0]["awards"]


[{'id': '2428016-SIE-HEP-2024-00115',
  'date': '2025-02-28T20:00:35-05:00',
  'items': [{'id': '4263344-DS-SO',
    'unit': {'id': '436', 'name': 'Unidad', 'scheme': 'SERCOP'},
    'quantity': 6600,
    'description': 'INSUMOS DE USO GENERAL',
    'classification': {'id': '352901091',
     'scheme': 'CPC',
     'description': 'INSUMOS DE USO GENERAL'},
    'additionalClassifications': [{'id': '35290.10.9',
      'uri': 'https://www.compraspublicas.gob.ec/ProcesoContratacion/compras/exe/verComProductos_exe.php?tipo=buscar&idProducto=35290.10.9',
      'scheme': 'CPC',
      'description': 'INSUMOS MEDICOS HOSPITALARIOS'}]}],
  'value': {'amount': 243500, 'currency': 'USD'},
  'suppliers': [{'id': 'EC-RUC-1312665100001-1217500',
    'name': 'ZAMBRANO VELEZ JEAN PIERRE'}],
  'description': 'Se resuelve ADJUDICAR el proceso de Subasta Inversa Electrónica Nro. SIE-HEP-2024-00115, cuyo objeto es la "ADQUISICIÓN DE DISPOSITIVOS MÉDICOS - EQUIPO DE PROTECCIÓN PERSONAL PARA LA ATENCIÓN OPORTUN

In [17]:
datos[2]["releases"][0]["awards"][0]["items"][0]["classification"]

{'id': '352901091', 'scheme': 'CPC', 'description': 'INSUMOS DE USO GENERAL'}

In [18]:
datos[2]["releases"][0]["tender"]["lots"]

[{'id': '432764',
  'title': 'OTROS PRODUCTOS O ARTICULOS FARMACEUTICOS PARA USOS MEDICOS O QUIRURGICOS',
  'value': {'amount': 260034, 'currency': 'USD'},
  'techniques': {'hasElectronicAuction': True}}]

In [19]:
datos[2]["releases"][0]["date"]

'2025-06-12T11:05:29-05:00'

In [20]:
datos[2]["releases"][0]["parties"][:2]

[{'id': 'EC-RUC-1360086920001-889479',
  'name': 'HOSPITAL DE ESPECIALIDADES PORTOVIEJO',
  'roles': ['buyer', 'procuringEntity'],
  'address': {'region': 'MANABI',
   'locality': 'PORTOVIEJO',
   'countryName': 'ECUADOR',
   'streetAddress': 'calle 15 de abril via santa ana s/n SAN PABLO'},
  'identifier': {'id': 'EC-RUC-1360086920001-889479',
   'scheme': 'EC-RUC',
   'legalName': 'HOSPITAL DE ESPECIALIDADES PORTOVIEJO'},
  'contactPoint': {'name': 'HOSPITAL DE ESPECIALIDADES PORTOVIEJO'}},
 {'id': 'EC-RUC-0190494071001-1114180',
  'name': 'TEXTILES E IMPORTACIONES PEZCAL CIA.LTDA.',
  'roles': ['tenderer'],
  'address': {'region': 'AZUAY',
   'locality': 'CUENCA',
   'postalCode': '010105',
   'countryName': 'ECUADOR',
   'streetAddress': 'TARQUINO MARTINEZ BORRERO FELIPE SERRANO S/N BELLAVISTA'},
  'identifier': {'id': 'EC-RUC-0190494071001-1114180',
   'scheme': 'EC-RUC',
   'legalName': 'TEXTILES E IMPORTACIONES PEZCAL CIA.LTDA.'},
  'contactPoint': {'name': 'TEXTILES E IMPORTACI